# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiknaxTheGreek/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [1]:
# Setup for the fixed FlyRank warehouse release.
# The token is read at runtime; never paste it into the notebook source.

import os
import duckdb
import pandas as pd

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN is missing. Add a Hugging Face READ token as a Colab Secret named HF_TOKEN."
    )

con = duckdb.connect()
safe_token = HF_TOKEN.replace("'", "''")
con.execute(
    f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{safe_token}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)
APRIL = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-04/*.parquet'"
    f")"
)

print("Warehouse paths ready.")
print("Feature window: March 1-31, 2026")
print("Outcome window: April 1-30, 2026")


Warehouse paths ready.
Feature window: March 1-31, 2026
Outcome window: April 1-30, 2026


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
# EXACTLY THREE formal verification queries for the Assignment 4 contract.

# Query 1 — raw grain: zero rows means no duplicate
# (report_date, client_hash_id, content_hash_id) keys.
grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM {MARCH}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print("QUERY 1 — grain check")
display(grain_check)


# Query 2 — size and date span of the March feature partition.
march_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS pages,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM {MARCH}
""").df()

print("\nQUERY 2 — March size and date span")
display(march_check)


# Query 3 — availability. Use IS TRUE because availability flags are
# not safely treated as ordinary two-valued booleans.
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
              AND ga4_data_available IS TRUE
        ) AS both_available_rows
    FROM {MARCH}
""").df()

print("\nQUERY 3 — March availability")
display(availability_check)


QUERY 1 — grain check


,report_date,client_hash_id,content_hash_id,row_count



QUERY 2 — March size and date span


,total_rows,clients,pages,first_date,last_date
0,9841378,55,331437,2026-03-01,2026-03-31



QUERY 3 — March availability


,total_rows,gsc_available_rows,ga4_available_rows,both_available_rows
0,9841378,3611061,413966,364347


### Design analysis 1 — determine the minimum usable-day requirement

This analysis is **not a fourth formal verification query**. It is a design-analysis block used to determine the coverage rule from the exact `FlyRank/internship-warehouse` March and April partitions.

A **usable GSC day** means a day for which the page has a daily record with `gsc_data_available IS TRUE`. A page may have zero impressions on a valid observed day, so usable-day coverage is **not** defined as `gsc_impressions > 0`.

No minimum-day value is assumed in advance.

The procedure is:

1. measure the March usable-day distribution at page level;
2. compare several candidate March cutoffs;
3. measure how many clients and pages each cutoff retains;
4. independently inspect April outcome-window coverage;
5. choose a practical March feature-eligibility cutoff from the observed trade-off;
6. choose an April outcome-observability requirement separately.

The March rule defines whether a page has enough **past information** to enter the proof-of-concept population. The April rule does **not** determine which pages are sampled in March. It only determines whether a selected March page has enough future observations for its April outcome to be evaluated reliably.

After the cells below are executed on the exact warehouse, the selected values and the numerical evidence supporting them will be written here explicitly.


In [4]:
# STEP 1A — exact March page-level GSC coverage.
# This scans only the March partition of FlyRank/internship-warehouse.

march_coverage = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(DISTINCT report_date) AS march_usable_days
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

print(f"Pages with at least one usable March GSC day: {len(march_coverage):,}")
print(f"Clients represented: {march_coverage['client_hash_id'].nunique():,}")

print("\nMarch usable-day distribution summary:")
display(
    march_coverage["march_usable_days"]
    .describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90])
    .to_frame("march_usable_days")
)

print("\nExact frequency table:")
march_day_distribution = (
    march_coverage["march_usable_days"]
    .value_counts()
    .sort_index()
    .rename_axis("usable_days")
    .reset_index(name="pages")
)
march_day_distribution["share_pct"] = (
    100.0 * march_day_distribution["pages"] / len(march_coverage)
)
display(march_day_distribution)


Pages with at least one usable March GSC day: 176,738
Clients represented: 47

March usable-day distribution summary:


,march_usable_days
count,176738.000000
mean,20.431718
std,11.480153
min,1.000000
10%,2.000000
25%,9.000000
50%,26.000000
75%,31.000000
90%,31.000000
max,31.000000



Exact frequency table:


,usable_days,pages,share_pct
0,1,13321,7.537145
1,2,7723,4.369745
2,3,5347,3.025382
3,4,4359,2.466363
4,5,3658,2.069730
5,6,3515,1.988820
6,7,2926,1.655558
7,8,2785,1.575779
8,9,2585,1.462617
9,10,2229,1.261189


In [5]:
# STEP 1B — compare candidate March cutoffs.
# No cutoff is locked before this table is inspected.

candidate_cutoffs = [7, 14, 18, 20, 21, 24, 27, 28, 30, 31]

rows = []
for cutoff in candidate_cutoffs:
    eligible = march_coverage[
        march_coverage["march_usable_days"] >= cutoff
    ].copy()

    per_client = eligible.groupby("client_hash_id").size()

    rows.append({
        "minimum_march_days": cutoff,
        "pages_retained": len(eligible),
        "clients_retained": eligible["client_hash_id"].nunique(),
        "pct_observed_pages_retained": round(
            100.0 * len(eligible) / len(march_coverage), 2
        ),
        "median_pages_per_client": (
            float(per_client.median()) if len(per_client) else 0.0
        ),
        "smallest_client_page_count": (
            int(per_client.min()) if len(per_client) else 0
        ),
        "pct_of_march_window_required": round(
            100.0 * cutoff / 31, 2
        )
    })

march_cutoff_analysis = pd.DataFrame(rows)
display(march_cutoff_analysis)


,minimum_march_days,pages_retained,clients_retained,pct_observed_pages_retained,median_pages_per_client,smallest_client_page_count,pct_of_march_window_required
0,7,138815,43,78.54,987.0,1,22.58
1,14,121844,40,68.94,738.5,1,45.16
2,18,111162,39,62.90,571.0,1,58.06
3,20,106546,37,60.28,713.0,1,64.52
4,21,103225,37,58.41,678.0,1,67.74
5,24,96013,37,54.33,589.0,1,77.42
6,27,86742,37,49.08,504.0,1,87.10
7,28,83842,37,47.44,478.0,1,90.32
8,30,67880,36,38.41,234.5,1,96.77
9,31,61796,34,34.96,295.0,1,100.00


In [6]:
# STEP 1C — inspect April outcome observability separately.
# IMPORTANT: April information is NOT used to construct the March sample.
# It is used only to determine whether a future outcome can be evaluated.

april_coverage = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(DISTINCT report_date) AS april_usable_days
    FROM {APRIL}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

print(f"Pages with at least one usable April GSC day: {len(april_coverage):,}")
print(f"April clients represented: {april_coverage['client_hash_id'].nunique():,}")

print("\nApril usable-day distribution summary:")
display(
    april_coverage["april_usable_days"]
    .describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90])
    .to_frame("april_usable_days")
)

coverage_pair = march_coverage.merge(
    april_coverage,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)
coverage_pair["april_usable_days"] = (
    coverage_pair["april_usable_days"].fillna(0).astype(int)
)

joint_rows = []
for cutoff in candidate_cutoffs:
    march_selected = coverage_pair[
        coverage_pair["march_usable_days"] >= cutoff
    ]
    outcome_observable = march_selected[
        march_selected["april_usable_days"] >= cutoff
    ]

    joint_rows.append({
        "candidate_days": cutoff,
        "march_feature_eligible_pages": len(march_selected),
        "march_feature_eligible_clients": march_selected["client_hash_id"].nunique(),
        "of_those_april_observable_pages": len(outcome_observable),
        "of_those_april_observable_clients": outcome_observable["client_hash_id"].nunique(),
        "pct_of_march_selected_with_observable_april": round(
            100.0 * len(outcome_observable) / len(march_selected), 2
        ) if len(march_selected) else 0.0
    })

coverage_tradeoff = pd.DataFrame(joint_rows)
display(coverage_tradeoff)

print(
    "\nInterpretation rule: choose the March eligibility cutoff from March coverage; "
    "treat April coverage only as label/outcome observability, not as a March sampling filter."
)


Pages with at least one usable April GSC day: 194,760
April clients represented: 51

April usable-day distribution summary:


,april_usable_days
count,194760.000000
mean,20.030088
std,10.985802
min,1.000000
10%,2.000000
25%,9.000000
50%,25.000000
75%,30.000000
90%,30.000000
max,30.000000


,candidate_days,march_feature_eligible_pages,march_feature_eligible_clients,of_those_april_observable_pages,of_those_april_observable_clients,pct_of_march_selected_with_observable_april
0,7,138815,43,128040,43,92.24
1,14,121844,40,111354,40,91.39
2,18,111162,39,100427,39,90.34
3,20,106546,37,95633,37,89.76
4,21,103225,37,92120,37,89.24
5,24,96013,37,84146,37,87.64
6,27,86742,37,73918,36,85.22
7,28,83842,37,70248,36,83.79
8,30,67880,36,52124,33,76.79
9,31,61796,34,0,0,0.00



Interpretation rule: choose the March eligibility cutoff from March coverage; treat April coverage only as label/outcome observability, not as a March sampling filter.


In [7]:
# STEP 1D — verify exact cohort identity at the chosen 20-day rule.
# This is a design check, not one of the three formal verification queries.

march20 = coverage_pair[
    coverage_pair["march_usable_days"] >= 20
].copy()

labeled20 = march20[
    march20["april_usable_days"] >= 20
].copy()

march20_clients = set(march20["client_hash_id"])
labeled20_clients = set(labeled20["client_hash_id"])

print("March >=20 clients:", len(march20_clients))
print("April-observable clients within that March cohort:", len(labeled20_clients))
print("Exact same client IDs:", march20_clients == labeled20_clients)
print("March clients missing from labeled cohort:", len(march20_clients - labeled20_clients))
print("Unexpected April-only clients:", len(labeled20_clients - march20_clients))

march20_keys = set(zip(march20["client_hash_id"], march20["content_hash_id"]))
labeled20_keys = set(zip(labeled20["client_hash_id"], labeled20["content_hash_id"]))

print("\nMarch feature-eligible page keys:", len(march20_keys))
print("Same page keys with >=20 April days:", len(labeled20_keys))
print("Every labeled page comes from the March cohort:", labeled20_keys.issubset(march20_keys))
print("March pages without sufficient April observability:", len(march20_keys - labeled20_keys))


March >=20 clients: 37
April-observable clients within that March cohort: 37
Exact same client IDs: True
March clients missing from labeled cohort: 0
Unexpected April-only clients: 0

March feature-eligible page keys: 106546
Same page keys with >=20 April days: 95633


Every labeled page comes from the March cohort: True
March pages without sufficient April observability: 10913


### Design analysis 2 — derive absolute March exposure tiers

This analysis uses only the **95,633 matched longitudinal page IDs** established in Step 1. The exposure signal is each page's **average March GSC impressions per usable GSC day**:

[
	ext{March exposure}_i =
rac{sum_d 	ext{gsc impressions}_{i,d}}
{	ext{usable March GSC days}_i}.
]

The purpose of the Low / Medium / High exposure tiers is **sampling balance**, not to label page quality or success.

The thresholds must be:

- derived from the observed matched-cohort distribution rather than assumed beforehand;
- expressed as **absolute average-impressions-per-day cut points**;
- identical for every client;
- evaluated for whether they preserve broad client representation and leave enough pages in every tier for equal-per-client sampling.

Three distribution-based candidate schemes are compared below: pooled terciles, pooled quartile outer bands, and pooled 20th/80th-percentile outer bands. These candidates are diagnostics, not final thresholds until their client-level viability is inspected.


In [8]:
# STEP 2A — March average impressions/day for the exact matched cohort.

matched_keys = labeled20[["client_hash_id", "content_hash_id"]].drop_duplicates().copy()
con.register("matched_keys", matched_keys)

march_exposure = con.sql(f"""
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        COUNT(DISTINCT f.report_date) AS march_usable_days,
        SUM(f.gsc_impressions) AS march_total_impressions,
        SUM(f.gsc_impressions)::DOUBLE
            / COUNT(DISTINCT f.report_date) AS march_avg_impressions_per_day
    FROM {MARCH} AS f
    INNER JOIN matched_keys AS k
        USING (client_hash_id, content_hash_id)
    WHERE f.gsc_data_available IS TRUE
    GROUP BY f.client_hash_id, f.content_hash_id
""").df()

print("Matched page rows:", f"{len(march_exposure):,}")
print("Matched clients:", march_exposure["client_hash_id"].nunique())
print(
    "Exact key count preserved:",
    len(march_exposure) == len(matched_keys)
)
print(
    "All pages have >=20 usable March days:",
    bool((march_exposure["march_usable_days"] >= 20).all())
)

print("\nMarch average-impressions/day distribution:")
display(
    march_exposure["march_avg_impressions_per_day"]
    .describe(
        percentiles=[
            0.10, 0.20, 0.25, 1/3, 0.50, 2/3,
            0.75, 0.80, 0.90, 0.95, 0.99
        ]
    )
    .to_frame("march_avg_impressions_per_day")
)

zero_pages = int(
    (march_exposure["march_avg_impressions_per_day"] == 0).sum()
)
print(
    f"Pages with zero average March impressions/day: "
    f"{zero_pages:,} "
    f"({100 * zero_pages / len(march_exposure):.2f}%)"
)


Matched page rows: 95,633
Matched clients: 37
Exact key count preserved: True
All pages have >=20 usable March days: True

March average-impressions/day distribution:


,march_avg_impressions_per_day
count,95633.000000
mean,93.805213
std,233.556763
min,1.035714
10%,4.433333
20%,7.354839
25%,9.290323
33.3%,13.419355
50%,27.451613
66.7%,58.252503


Pages with zero average March impressions/day: 0 (0.00%)


In [9]:
# STEP 2B — create data-driven candidate absolute exposure thresholds.

x = march_exposure["march_avg_impressions_per_day"]

candidate_schemes = {
    "terciles_33_67": (
        float(x.quantile(1/3)),
        float(x.quantile(2/3))
    ),
    "quartile_25_75": (
        float(x.quantile(0.25)),
        float(x.quantile(0.75))
    ),
    "outer_20_80": (
        float(x.quantile(0.20)),
        float(x.quantile(0.80))
    )
}

threshold_rows = []
for scheme, (low_thr, high_thr) in candidate_schemes.items():
    tier = pd.cut(
        x,
        bins=[-float("inf"), low_thr, high_thr, float("inf")],
        labels=["Low", "Medium", "High"],
        include_lowest=True
    )

    counts = tier.value_counts().reindex(["Low", "Medium", "High"], fill_value=0)

    threshold_rows.append({
        "scheme": scheme,
        "low_max_avg_impressions_per_day": low_thr,
        "high_min_boundary_avg_impressions_per_day": high_thr,
        "low_pages": int(counts["Low"]),
        "medium_pages": int(counts["Medium"]),
        "high_pages": int(counts["High"]),
        "low_pct": round(100 * counts["Low"] / len(x), 2),
        "medium_pct": round(100 * counts["Medium"] / len(x), 2),
        "high_pct": round(100 * counts["High"] / len(x), 2)
    })

threshold_comparison = pd.DataFrame(threshold_rows)
display(threshold_comparison)


,scheme,low_max_avg_impressions_per_day,high_min_boundary_avg_impressions_per_day,low_pages,medium_pages,high_pages,low_pct,medium_pct,high_pct
0,terciles_33_67,13.419355,58.252503,31896,31859,31878,33.35,33.31,33.33
1,quartile_25_75,9.290323,88.724138,23935,47790,23908,25.03,49.97,25.00
2,outer_20_80,7.354839,117.387097,19134,57375,19124,20.01,59.99,20.00


In [10]:
# STEP 2C — client-level viability of each candidate threshold scheme.
# For equal-per-client sampling, a client must contribute pages to all three tiers.

viability_rows = []

for scheme, (low_thr, high_thr) in candidate_schemes.items():
    temp = march_exposure[
        ["client_hash_id", "content_hash_id", "march_avg_impressions_per_day"]
    ].copy()

    temp["exposure_tier"] = pd.cut(
        temp["march_avg_impressions_per_day"],
        bins=[-float("inf"), low_thr, high_thr, float("inf")],
        labels=["Low", "Medium", "High"],
        include_lowest=True
    )

    client_tiers = (
        temp.groupby(["client_hash_id", "exposure_tier"], observed=False)
        .size()
        .unstack(fill_value=0)
        .reindex(columns=["Low", "Medium", "High"], fill_value=0)
    )

    client_tiers["min_tier_pages"] = client_tiers[
        ["Low", "Medium", "High"]
    ].min(axis=1)

    viability_rows.append({
        "scheme": scheme,
        "clients_total": len(client_tiers),
        "clients_with_all_3_tiers": int((client_tiers["min_tier_pages"] >= 1).sum()),
        "clients_with_at_least_10_each": int((client_tiers["min_tier_pages"] >= 10).sum()),
        "clients_with_at_least_25_each": int((client_tiers["min_tier_pages"] >= 25).sum()),
        "clients_with_at_least_50_each": int((client_tiers["min_tier_pages"] >= 50).sum()),
        "clients_with_at_least_100_each": int((client_tiers["min_tier_pages"] >= 100).sum()),
        "median_of_client_min_tier_pages": float(client_tiers["min_tier_pages"].median()),
        "p25_of_client_min_tier_pages": float(client_tiers["min_tier_pages"].quantile(0.25)),
        "smallest_client_min_tier_pages": int(client_tiers["min_tier_pages"].min())
    })

viability_comparison = pd.DataFrame(viability_rows)
display(viability_comparison)

# Keep the per-client table for the pooled-tercile candidate visible as the
# strongest default balancing option unless the diagnostics contradict it.
tercile_low, tercile_high = candidate_schemes["terciles_33_67"]
tercile_page_tiers = march_exposure.copy()
tercile_page_tiers["exposure_tier"] = pd.cut(
    tercile_page_tiers["march_avg_impressions_per_day"],
    bins=[-float("inf"), tercile_low, tercile_high, float("inf")],
    labels=["Low", "Medium", "High"],
    include_lowest=True
)

tercile_client_counts = (
    tercile_page_tiers
    .groupby(["client_hash_id", "exposure_tier"], observed=False)
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["Low", "Medium", "High"], fill_value=0)
)
tercile_client_counts["min_tier_pages"] = tercile_client_counts.min(axis=1)

print("\nPooled-tercile per-client tier counts:")
display(
    tercile_client_counts
    .sort_values("min_tier_pages")
)


,scheme,clients_total,clients_with_all_3_tiers,clients_with_at_least_10_each,clients_with_at_least_25_each,clients_with_at_least_50_each,clients_with_at_least_100_each,median_of_client_min_tier_pages,p25_of_client_min_tier_pages,smallest_client_min_tier_pages
0,terciles_33_67,37,28,22,21,19,14,64.0,1.0,0
1,quartile_25_75,37,26,21,18,16,12,24.0,0.0,0
2,outer_20_80,37,25,21,17,12,11,15.0,0.0,0



Pooled-tercile per-client tier counts:


exposure_tier,Low,Medium,High,min_tier_pages
client_hash_id,,,,
client_0e1acc6cd57b0eba,4,0,0,0
client_2b4306c3ed003f01,7,0,0,0
client_59256b0571e0c970,1,0,0,0
client_def0955f7a377868,21,3,0,0
client_8ae2bfb5aa1ffa1e,4,0,0,0
client_8dbf3abdf07569e0,4,0,0,0
client_9d54435aabd95a6c,3,0,0,0
client_795153d5b7850ccf,36,6,0,0
client_d211cb07b9059bab,13,5,0,0


In [11]:
# STEP 2D — sensitivity check for readable operational cut points.

exact_low, exact_high = candidate_schemes["terciles_33_67"]

rounded_candidates = {
    "exact_terciles": (exact_low, exact_high),
    "rounded_2dp": (round(exact_low, 2), round(exact_high, 2)),
    "rounded_1dp": (round(exact_low, 1), round(exact_high, 1)),
    "rounded_integer": (round(exact_low), round(exact_high))
}

rounding_rows = []

for label, (low_thr, high_thr) in rounded_candidates.items():
    temp = march_exposure[
        ["client_hash_id", "content_hash_id", "march_avg_impressions_per_day"]
    ].copy()

    temp["exposure_tier"] = pd.cut(
        temp["march_avg_impressions_per_day"],
        bins=[-float("inf"), low_thr, high_thr, float("inf")],
        labels=["Low", "Medium", "High"],
        include_lowest=True
    )

    counts = temp["exposure_tier"].value_counts().reindex(
        ["Low", "Medium", "High"], fill_value=0
    )

    by_client = (
        temp.groupby(["client_hash_id", "exposure_tier"], observed=False)
        .size()
        .unstack(fill_value=0)
        .reindex(columns=["Low", "Medium", "High"], fill_value=0)
    )
    by_client["min_tier_pages"] = by_client.min(axis=1)

    rounding_rows.append({
        "threshold_version": label,
        "low_max": low_thr,
        "high_boundary": high_thr,
        "low_pages": int(counts["Low"]),
        "medium_pages": int(counts["Medium"]),
        "high_pages": int(counts["High"]),
        "clients_with_all_3_tiers": int((by_client["min_tier_pages"] >= 1).sum()),
        "clients_with_at_least_10_each": int((by_client["min_tier_pages"] >= 10).sum()),
        "clients_with_at_least_25_each": int((by_client["min_tier_pages"] >= 25).sum()),
        "clients_with_at_least_50_each": int((by_client["min_tier_pages"] >= 50).sum())
    })

rounding_sensitivity = pd.DataFrame(rounding_rows)
display(rounding_sensitivity)


,threshold_version,low_max,high_boundary,low_pages,medium_pages,high_pages,clients_with_all_3_tiers,clients_with_at_least_10_each,clients_with_at_least_25_each,clients_with_at_least_50_each
0,exact_terciles,13.419355,58.252503,31896,31859,31878,28,22,21,19
1,rounded_2dp,13.420000,58.250000,31896,31859,31878,28,22,21,19
2,rounded_1dp,13.400000,58.300000,31855,31924,31854,28,22,21,19
3,rounded_integer,13.000000,58.000000,31186,32489,31958,28,22,21,19


#### Step 2 decision — locked exposure thresholds

The pooled **33.3rd and 66.7th percentiles** of March average impressions per usable day are approximately **13.419355** and **58.252503**. Because the purpose is to create three exposure strata for balanced sampling, pooled terciles are directly aligned with the three-tier design.

The sensitivity analysis showed that rounding those boundaries to **13.42** and **58.25 impressions per usable day** produces **exactly the same tier counts and the same client-level viability** as the full-precision cut points. The readable two-decimal thresholds are therefore used operationally.

The locked exposure tiers are:

- **Low exposure:** March average impressions/day ≤ **13.42**
- **Medium exposure:** > **13.42** and ≤ **58.25**
- **High exposure:** > **58.25**

These are **absolute thresholds shared by every client**. They describe exposure volume only; they are not performance, quality, or outcome labels.

On the 95,633-page matched cohort, this yields approximately equal global thirds. The pooled-tercile scheme also preserved the broadest cross-client viability among the tested distribution-based alternatives. The exact number of clients retained for the final balanced proof-of-concept sample, and the equal pages-per-tier-per-client sample size, are determined in the next design step rather than forced here.


In [12]:
# STEP 2E — apply the locked absolute exposure thresholds.

EXPOSURE_LOW_MAX = 13.42
EXPOSURE_MEDIUM_MAX = 58.25

exposure_frame = march_exposure.copy()
exposure_frame["exposure_tier"] = pd.cut(
    exposure_frame["march_avg_impressions_per_day"],
    bins=[
        -float("inf"),
        EXPOSURE_LOW_MAX,
        EXPOSURE_MEDIUM_MAX,
        float("inf")
    ],
    labels=["Low", "Medium", "High"],
    include_lowest=True
)

locked_tier_counts = (
    exposure_frame["exposure_tier"]
    .value_counts()
    .reindex(["Low", "Medium", "High"], fill_value=0)
)

locked_client_tiers = (
    exposure_frame
    .groupby(["client_hash_id", "exposure_tier"], observed=False)
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["Low", "Medium", "High"], fill_value=0)
)
locked_client_tiers["min_tier_pages"] = locked_client_tiers.min(axis=1)

print(f"Locked Low maximum: {EXPOSURE_LOW_MAX:.2f} impressions/day")
print(f"Locked Medium maximum: {EXPOSURE_MEDIUM_MAX:.2f} impressions/day")
print("\nTier counts:")
display(locked_tier_counts.to_frame("pages"))

print("Total pages preserved:", int(locked_tier_counts.sum()))
print("Expected matched pages:", len(matched_keys))
print("Exact matched population preserved:", int(locked_tier_counts.sum()) == len(matched_keys))
print("Clients represented anywhere:", exposure_frame["client_hash_id"].nunique())
print("Clients represented in all three tiers:", int((locked_client_tiers["min_tier_pages"] >= 1).sum()))


Locked Low maximum: 13.42 impressions/day
Locked Medium maximum: 58.25 impressions/day

Tier counts:


,pages
exposure_tier,
Low,31896
Medium,31859
High,31878


Total pages preserved: 95633
Expected matched pages: 95633
Exact matched population preserved: True
Clients represented anywhere: 37
Clients represented in all three tiers: 28


### Design analysis 3 — choose the balanced client/page sample

The final proof-of-concept population must prevent large clients from dominating the analysis. Starting from the locked exposure tiers, a client is eligible for balanced sampling only if it contains pages in **all three** exposure tiers.

For every candidate pages-per-tier cap below:

- the same cap is applied to every retained client;
- the same number of Low, Medium, and High pages is sampled within each retained client;
- client breadth is treated as more important than simply maximizing the total page count;
- the final page sample remains a subset of the already matched March→April longitudinal cohort.

The cap is determined from the observed client-tier availability trade-off rather than chosen in advance.


In [13]:
# STEP 3A — inspect the distribution of the limiting tier within each client.

three_tier_clients = locked_client_tiers[
    locked_client_tiers["min_tier_pages"] >= 1
].copy()

print("Clients with all three exposure tiers:", len(three_tier_clients))
print("\nDistribution of each client's smallest tier:")
display(
    three_tier_clients["min_tier_pages"]
    .describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90])
    .to_frame("min_tier_pages")
)

print("\nEligible clients ordered by their limiting tier:")
display(
    three_tier_clients[["Low", "Medium", "High", "min_tier_pages"]]
    .sort_values("min_tier_pages")
)


Clients with all three exposure tiers: 28

Distribution of each client's smallest tier:


,min_tier_pages
count,28.000000
mean,803.607143
std,1587.397155
min,1.000000
10%,2.000000
25%,33.250000
50%,101.000000
75%,596.750000
90%,2134.300000
max,6875.000000



Eligible clients ordered by their limiting tier:


exposure_tier,Low,Medium,High,min_tier_pages
client_hash_id,,,,
client_08d2847f24cf89c1,13,3,1,1
client_ccdd78843409c8c7,6,3,1,1
client_cd12bcfd98942aa1,76,29,2,2
client_0797ff3a1fc9a6a5,6,2,3,2
client_3ffa76342f366962,91,18,4,4
client_a2eeb8899886adde,4,4,4,4
client_f623b01661d4bfe4,58,41,10,10
client_400c21c81c8b46ef,393,182,41,41
client_b10cb2997d0c7c86,190,95,41,41


In [14]:
# STEP 3B — compare candidate equal pages-per-tier-per-client caps.

candidate_caps = [5, 10, 15, 20, 25, 30, 40, 50, 75, 100, 150, 200, 250, 500]

cap_rows = []
for cap in candidate_caps:
    eligible_clients = three_tier_clients[
        three_tier_clients["min_tier_pages"] >= cap
    ]

    n_clients = len(eligible_clients)
    total_pages = n_clients * 3 * cap

    cap_rows.append({
        "pages_per_tier_per_client": cap,
        "clients_retained": n_clients,
        "pct_of_3tier_clients_retained": round(
            100.0 * n_clients / len(three_tier_clients), 2
        ),
        "balanced_pages_per_client": 3 * cap,
        "total_balanced_pages": total_pages,
        "low_pages": n_clients * cap,
        "medium_pages": n_clients * cap,
        "high_pages": n_clients * cap
    })

cap_tradeoff = pd.DataFrame(cap_rows)
display(cap_tradeoff)

# Marginal trade-off relative to the previous candidate.
cap_tradeoff_marginal = cap_tradeoff.copy()
cap_tradeoff_marginal["clients_lost_vs_previous"] = (
    cap_tradeoff_marginal["clients_retained"]
    .shift(1)
    .sub(cap_tradeoff_marginal["clients_retained"])
)
cap_tradeoff_marginal["pages_gained_vs_previous"] = (
    cap_tradeoff_marginal["total_balanced_pages"]
    .diff()
)
print("\nMarginal trade-off:")
display(cap_tradeoff_marginal)


,pages_per_tier_per_client,clients_retained,pct_of_3tier_clients_retained,balanced_pages_per_client,total_balanced_pages,low_pages,medium_pages,high_pages
0,5,22,78.57,15,330,110,110,110
1,10,22,78.57,30,660,220,220,220
2,15,21,75.00,45,945,315,315,315
3,20,21,75.00,60,1260,420,420,420
4,25,21,75.00,75,1575,525,525,525
5,30,21,75.00,90,1890,630,630,630
6,40,21,75.00,120,2520,840,840,840
7,50,19,67.86,150,2850,950,950,950
8,75,16,57.14,225,3600,1200,1200,1200
9,100,14,50.00,300,4200,1400,1400,1400



Marginal trade-off:


,pages_per_tier_per_client,clients_retained,pct_of_3tier_clients_retained,balanced_pages_per_client,total_balanced_pages,low_pages,medium_pages,high_pages,clients_lost_vs_previous,pages_gained_vs_previous
0,5,22,78.57,15,330,110,110,110,NaN,NaN
1,10,22,78.57,30,660,220,220,220,0.0,330.0
2,15,21,75.00,45,945,315,315,315,1.0,285.0
3,20,21,75.00,60,1260,420,420,420,0.0,315.0
4,25,21,75.00,75,1575,525,525,525,0.0,315.0
5,30,21,75.00,90,1890,630,630,630,0.0,315.0
6,40,21,75.00,120,2520,840,840,840,0.0,630.0
7,50,19,67.86,150,2850,950,950,950,2.0,330.0
8,75,16,57.14,225,3600,1200,1200,1200,3.0,750.0
9,100,14,50.00,300,4200,1400,1400,1400,2.0,600.0


#### Step 3 decision — locked balanced proof-of-concept sample

The candidate-cap analysis shows a clear **client-breadth plateau**:

- 15, 20, 25, 30, and **40** pages per tier each retain the same **21 clients**;
- raising the cap from 30 to 40 therefore increases the balanced sample without losing any additional client;
- raising the cap from **40 to 50** increases the sample by only **330 pages** but removes **2 clients**, reducing client coverage from 21 to 19.

Because the population design prioritizes broad client representation before additional within-client depth, the final cap is therefore set to **40 pages per exposure tier per retained client**.

This produces:

- **21 clients**
- **40 Low + 40 Medium + 40 High pages per client**
- **120 pages per client**
- **840 pages per exposure tier globally**
- **2,520 pages total**

The 21 retained clients are precisely those with at least 40 matched longitudinal pages in every exposure tier. Page selection within each client × tier cell is deterministic: rows are ordered by the already-hashed `content_hash_id` and the first 40 are retained. This avoids random-run drift while using no outcome information to choose among otherwise eligible pages.

The final balanced population remains entirely inside the exact matched March→April cohort, so every selected `(client_hash_id, content_hash_id)` has sufficient observations in both months.


In [15]:
# STEP 3C — construct and verify the locked balanced POC population.

PAGES_PER_TIER_PER_CLIENT = 40

final_clients = (
    locked_client_tiers[
        locked_client_tiers["min_tier_pages"] >= PAGES_PER_TIER_PER_CLIENT
    ]
    .index
    .tolist()
)

balanced_poc = (
    exposure_frame[
        exposure_frame["client_hash_id"].isin(final_clients)
    ]
    .sort_values(
        ["client_hash_id", "exposure_tier", "content_hash_id"]
    )
    .groupby(
        ["client_hash_id", "exposure_tier"],
        observed=False,
        group_keys=False
    )
    .head(PAGES_PER_TIER_PER_CLIENT)
    .reset_index(drop=True)
)

# Verification summaries.
per_client_tier = (
    balanced_poc
    .groupby(["client_hash_id", "exposure_tier"], observed=False)
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["Low", "Medium", "High"], fill_value=0)
)

tier_totals = (
    balanced_poc["exposure_tier"]
    .value_counts()
    .reindex(["Low", "Medium", "High"], fill_value=0)
)

balanced_keys = set(zip(
    balanced_poc["client_hash_id"],
    balanced_poc["content_hash_id"]
))
matched_key_set = set(zip(
    matched_keys["client_hash_id"],
    matched_keys["content_hash_id"]
))

print("Final clients:", balanced_poc["client_hash_id"].nunique())
print("Final pages:", len(balanced_poc))
print("Unique client-page keys:", len(balanced_keys))
print("Expected pages:", 21 * 3 * PAGES_PER_TIER_PER_CLIENT)
print("All selected keys are in matched March-April cohort:", balanced_keys.issubset(matched_key_set))
print("Every client has exactly 40 pages in every tier:", bool((per_client_tier == 40).all().all()))
print("Every client contributes exactly 120 pages:", bool((per_client_tier.sum(axis=1) == 120).all()))

print("\nGlobal tier totals:")
display(tier_totals.to_frame("pages"))

print("\nPer-client tier balance:")
display(per_client_tier)

# Audit the exact client identities retained.
print("\nExact retained client IDs:")
for client_id in sorted(final_clients):
    print(client_id)


Final clients: 21
Final pages: 2520
Unique client-page keys: 2520
Expected pages: 2520
All selected keys are in matched March-April cohort: True
Every client has exactly 40 pages in every tier: True
Every client contributes exactly 120 pages: True

Global tier totals:


,pages
exposure_tier,
Low,840
Medium,840
High,840



Per-client tier balance:


exposure_tier,Low,Medium,High
client_hash_id,,,
client_08a6a72ff48e62c0,40,40,40
client_0fa64a184f18a4a0,40,40,40
client_157ffe4d4a595515,40,40,40
client_1a730cb2640a1abf,40,40,40
client_20259bd6705d81d4,40,40,40
client_2094c6eb080311d5,40,40,40
client_23a62021009f63c4,40,40,40
client_3197e6291363b4db,40,40,40
client_3f0ce4d44fe94f3d,40,40,40



Exact retained client IDs:
client_08a6a72ff48e62c0
client_0fa64a184f18a4a0
client_157ffe4d4a595515
client_1a730cb2640a1abf
client_20259bd6705d81d4
client_2094c6eb080311d5
client_23a62021009f63c4
client_3197e6291363b4db
client_3f0ce4d44fe94f3d
client_400c21c81c8b46ef
client_62f4a7e64f5e0096
client_65de48885f4ef01b
client_73cda7b4e4f265ea
client_9958f0a7ae1df715
client_a80fca3f171ed1de
client_b10cb2997d0c7c86
client_c182d11e4862a37d
client_e547b89c05043229
client_e5c2aa26a8598242
client_fef1a8f436438636
client_ff644d8251367cbb


### Design analysis 4 — construct and audit the five March features

The assignment permits **at most five features**. The feature set is therefore intentionally restricted to five March-only quantities, each representing a different behavioral dimension:

1. **Exposure:** average GSC impressions per usable March day.
2. **Engagement:** aggregate March CTR, computed as total clicks / total impressions.
3. **Visibility:** impression-weighted March average GSC position.
4. **Direction:** the linear slope of daily impressions across March, normalized by that page's average daily impressions.
5. **Stability:** the coefficient of variation of daily March impressions.

All five are computed only from `gsc_data_available IS TRUE` rows for the already-locked 2,520-page balanced population. No April value enters any feature.

The normalized trend and coefficient of variation are used instead of raw slope and raw standard deviation because raw magnitude measures would largely re-encode exposure: a page receiving thousands of impressions naturally has larger absolute changes than a page receiving tens. Normalizing by the page's own March mean separates **direction** and **relative instability** from absolute exposure.

The code below checks row preservation, missing/non-finite values, distributions, and pairwise Spearman correlations before the feature set is locked.


In [16]:
# STEP 4A — build the exact five-feature March frame for the locked 2,520 pages.

import numpy as np

balanced_keys_df = balanced_poc[
    ["client_hash_id", "content_hash_id", "exposure_tier"]
].copy()
con.register("balanced_keys_df", balanced_keys_df)

feature_frame = con.sql(f"""
    WITH daily AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,
            f.report_date,
            DATE_DIFF(
                'day',
                DATE '2026-03-01',
                f.report_date
            )::DOUBLE AS march_day_index,
            f.gsc_impressions::DOUBLE AS impressions,
            f.gsc_clicks::DOUBLE AS clicks,
            f.gsc_avg_position::DOUBLE AS avg_position
        FROM {MARCH} AS f
        INNER JOIN balanced_keys_df AS k
            USING (client_hash_id, content_hash_id)
        WHERE f.gsc_data_available IS TRUE
    )
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(impressions)
            / COUNT(DISTINCT report_date)
            AS avg_impressions_per_day,

        SUM(clicks)
            / NULLIF(SUM(impressions), 0)
            AS aggregate_ctr,

        SUM(avg_position * impressions)
            / NULLIF(SUM(impressions), 0)
            AS weighted_avg_position,

        REGR_SLOPE(impressions, march_day_index)
            / NULLIF(AVG(impressions), 0)
            AS relative_impression_trend_per_day,

        STDDEV_SAMP(impressions)
            / NULLIF(AVG(impressions), 0)
            AS impression_cv

    FROM daily
    GROUP BY client_hash_id, content_hash_id
""").df()

feature_names = [
    "avg_impressions_per_day",
    "aggregate_ctr",
    "weighted_avg_position",
    "relative_impression_trend_per_day",
    "impression_cv"
]

print("Feature rows:", len(feature_frame))
print("Expected rows:", len(balanced_poc))
print(
    "Exact 2,520-page population preserved:",
    set(zip(feature_frame["client_hash_id"], feature_frame["content_hash_id"]))
    ==
    set(zip(balanced_poc["client_hash_id"], balanced_poc["content_hash_id"]))
)
print("Number of model features:", len(feature_names))

print("\nSmall March feature frame:")
display(feature_frame.head(10))


Feature rows: 2520
Expected rows: 2520
Exact 2,520-page population preserved: True
Number of model features: 5

Small March feature frame:


,client_hash_id,content_hash_id,avg_impressions_per_day,aggregate_ctr,weighted_avg_position,relative_impression_trend_per_day,impression_cv
0,client_400c21c81c8b46ef,content_4a8757337de2cc1f,289.290323,0.001450,3.731936,-0.003970,0.359621
1,client_400c21c81c8b46ef,content_2f37ea886f689909,52.032258,0.000620,5.921885,0.002705,0.468863
2,client_400c21c81c8b46ef,content_382ce19bd9061566,208.000000,0.002171,3.493331,-0.016115,0.377188
3,client_400c21c81c8b46ef,content_073eeab0107ad398,217.806452,0.006368,8.138033,-0.010456,0.387342
4,client_400c21c81c8b46ef,content_024a20ae8d43fcec,115.612903,0.000837,7.636440,0.062336,0.706324
5,client_400c21c81c8b46ef,content_26d8e18b7b46cdd6,56.483871,0.000000,4.335237,-0.006646,0.408320
6,client_400c21c81c8b46ef,content_336a7ee8b7540d61,32.580645,0.000990,7.576238,0.025730,0.492989
7,client_400c21c81c8b46ef,content_0de6580ab609900b,7.967742,0.000000,7.012146,0.018978,0.490902
8,client_400c21c81c8b46ef,content_34d38ec3df1c9e67,87.935484,0.001467,6.724505,0.077210,0.842084
9,client_400c21c81c8b46ef,content_08ae2414078abc20,32.612903,0.001978,5.433234,0.035546,0.458759


In [17]:
# STEP 4B — audit completeness, finiteness, distributions, and redundancy.

missingness = pd.DataFrame({
    "feature": feature_names,
    "missing_n": [int(feature_frame[c].isna().sum()) for c in feature_names],
    "missing_pct": [
        round(100.0 * feature_frame[c].isna().mean(), 4)
        for c in feature_names
    ],
    "non_finite_n": [
        int((~np.isfinite(feature_frame[c].astype(float))).sum())
        for c in feature_names
    ]
})
print("Missingness / finiteness:")
display(missingness)

print("\nFeature distributions:")
display(
    feature_frame[feature_names]
    .describe(percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])
    .T
)

spearman_corr = feature_frame[feature_names].corr(method="spearman")
print("\nSpearman feature correlations:")
display(spearman_corr.round(3))

corr_abs = spearman_corr.abs().copy(deep=True)
for feature in feature_names:
    corr_abs.loc[feature, feature] = np.nan
max_pair = corr_abs.stack().idxmax()
max_value = corr_abs.stack().max()

print(
    f"\nLargest absolute off-diagonal Spearman correlation: "
    f"{max_pair[0]} vs {max_pair[1]} = {max_value:.3f}"
)
print(
    "Any absolute pairwise Spearman correlation >= 0.90:",
    bool((corr_abs.stack() >= 0.90).any())
)

# Confirm that exposure in the feature frame exactly reproduces the locked
# March exposure calculation for the same page IDs.
exposure_check = feature_frame.merge(
    balanced_poc[
        ["client_hash_id", "content_hash_id", "march_avg_impressions_per_day"]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

print(
    "Exposure feature exactly matches locked exposure calculation:",
    bool(np.allclose(
        exposure_check["avg_impressions_per_day"],
        exposure_check["march_avg_impressions_per_day"],
        rtol=0,
        atol=1e-12
    ))
)


Missingness / finiteness:


,feature,missing_n,missing_pct,non_finite_n
0,avg_impressions_per_day,0,0.0,0
1,aggregate_ctr,0,0.0,0
2,weighted_avg_position,0,0.0,0
3,relative_impression_trend_per_day,0,0.0,0
4,impression_cv,0,0.0,0



Feature distributions:


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
avg_impressions_per_day,2520.0,73.262470,170.257601,1.350000,2.333333,3.583013,9.838710,26.806452,74.846774,259.641935,705.636774,4091.483871
aggregate_ctr,2520.0,0.002862,0.004459,0.000000,0.000000,0.000000,0.000000,0.001302,0.004028,0.010813,0.022637,0.051192
weighted_avg_position,2520.0,12.768534,13.050539,0.284757,1.423998,2.615830,5.403486,7.633538,14.712566,40.191396,67.571000,85.110000
relative_impression_trend_per_day,2520.0,-0.001015,0.037235,-0.180663,-0.106385,-0.058958,-0.022048,-0.001648,0.019261,0.064444,0.094160,0.163514
impression_cv,2520.0,0.605079,0.322016,0.122238,0.199459,0.263360,0.402993,0.540560,0.719371,1.173604,1.834392,3.809847



Spearman feature correlations:


,avg_impressions_per_day,aggregate_ctr,weighted_avg_position,relative_impression_trend_per_day,impression_cv
avg_impressions_per_day,1.000,0.311,-0.299,0.058,-0.382
aggregate_ctr,0.311,1.000,-0.304,0.071,-0.189
weighted_avg_position,-0.299,-0.304,1.000,-0.015,-0.015
relative_impression_trend_per_day,0.058,0.071,-0.015,1.000,-0.024
impression_cv,-0.382,-0.189,-0.015,-0.024,1.000



Largest absolute off-diagonal Spearman correlation: avg_impressions_per_day vs impression_cv = 0.382
Any absolute pairwise Spearman correlation >= 0.90: False
Exposure feature exactly matches locked exposure calculation: True


In [ ]:
# STEP 4C — sanity-check the warehouse position metric before locking it.

position_sanity = con.sql(f"""
    SELECT
        COUNT(*) AS usable_daily_rows,
        COUNT(*) FILTER (WHERE f.gsc_impressions > 0) AS positive_impression_rows,
        COUNT(*) FILTER (
            WHERE f.gsc_impressions > 0
              AND f.gsc_avg_position < 1
        ) AS positive_impression_rows_position_below_1,
        MIN(f.gsc_avg_position) FILTER (
            WHERE f.gsc_impressions > 0
        ) AS min_position_when_impressions_positive,
        MAX(f.gsc_avg_position) FILTER (
            WHERE f.gsc_impressions > 0
        ) AS max_position_when_impressions_positive
    FROM {MARCH} AS f
    INNER JOIN balanced_keys_df AS k
        USING (client_hash_id, content_hash_id)
    WHERE f.gsc_data_available IS TRUE
""").df()

print("Daily position sanity check:")
display(position_sanity)

position_compare = con.sql(f"""
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        SUM(f.gsc_avg_position * f.gsc_impressions)
            / NULLIF(SUM(f.gsc_impressions), 0)
            AS weighted_from_daily_avg_position,
        SUM(f.gsc_sum_position)
            / NULLIF(SUM(f.gsc_impressions), 0)
            AS position_from_sum_position
    FROM {MARCH} AS f
    INNER JOIN balanced_keys_df AS k
        USING (client_hash_id, content_hash_id)
    WHERE f.gsc_data_available IS TRUE
    GROUP BY f.client_hash_id, f.content_hash_id
""").df()

position_compare["abs_difference"] = (
    position_compare["weighted_from_daily_avg_position"]
    - position_compare["position_from_sum_position"]
).abs()

print("\nComparison with gsc_sum_position / impressions:")
display(
    position_compare[
        [
            "weighted_from_daily_avg_position",
            "position_from_sum_position",
            "abs_difference"
        ]
    ].describe(percentiles=[0.50, 0.95, 0.99]).T
)

print(
    "Max absolute difference between the two monthly constructions:",
    float(position_compare["abs_difference"].max())
)


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.